# Harness vs training: full run

Settings: Accelerator **GPU T4**, Internet **on**. Use **Save Version > Save & Run All (Commit)** so it runs in the background (about 3 hours). Outputs land in `/kaggle/working/harness-vs-training/results`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
%cd /kaggle/working
!git clone -q https://github.com/SvshSingh/harness-vs-training.git
%cd harness-vs-training
!git log --oneline -1
!pip install -q -U "transformers>=4.56" "peft>=0.13" accelerate pytest

In [ ]:
!python -m pytest -q
# 32-example smoke test before committing hours of GPU time
!python -m hvt.evaluate --mode harness --limit 32 --out results/smoke/harness 2>&1 | grep -v -i warn | tail -12

In [ ]:
!rm -rf results/smoke
!bash scripts/run_all.sh 2>&1 | grep -v -i -E 'warn|Loading weights' 

In [ ]:
from IPython.display import Image, Markdown, display
display(Markdown(open('results/table.md').read()))
display(Image('results/accuracy.png'))
!mkdir -p results/train_meta && for d in runs/*/; do cp "$d/train_meta.json" "results/train_meta/$(basename $d).json"; done
!cp data/weights.json results/ && cd /kaggle/working && zip -qr results.zip harness-vs-training/results && ls -lh results.zip